# Streaming Algorithm Analysis: Flajolet-Martin & Count-Min Sketch

---

## Algorithm Choices

| Algorithm | What it estimates |
|-----------|-------------------|
| **Flajolet-Martin (FM)** — 3 versions: Alpha, Beta, Final | F₀ — distinct element count |
| **Count-Min Sketch (CMS)** | Point frequency (biased overestimate) |

### Overview

**FM (Flajolet-Martin)** estimates the number of distinct elements (F₀) in a data stream using random hashing to [0,1]. We implement three progressively better versions:
1. **Alpha** — Single estimator, high variance
2. **Beta** — Average of s independent copies, variance reduced by factor s
3. **Final** — Median of t Beta groups, exponential tail concentration

**Count-Min Sketch** estimates point frequencies using a d×w hash table. It always overestimates (biased), with error bounded by ε·F₁ with probability ≥ 1−δ.

In [1]:
# ============================================================
# Cell 2: Imports & Configuration
# ============================================================
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from scipy.stats import zipf as zipf_dist
import math
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Experiment parameters ──
N_STREAM = 200_000          # Total stream length (≥100K required)
VOCAB_SIZE = 15_000         # Vocabulary size for Zipf generation
ZIPF_PARAM = 1.5            # Zipf exponent (controls skewness)
N_REPETITIONS = 100         # Outer repetitions per experiment (≥50 required)
DELTA_TARGET = 0.05         # Target failure probability (95% success)

print(f"Stream length N = {N_STREAM:,}")
print(f"Vocabulary size  = {VOCAB_SIZE:,}")
print(f"Zipf parameter a = {ZIPF_PARAM}")
print(f"Repetitions      = {N_REPETITIONS}")
print(f"Target δ         = {DELTA_TARGET}  (95% success)")

Stream length N = 200,000
Vocabulary size  = 15,000
Zipf parameter a = 1.5
Repetitions      = 100
Target δ         = 0.05  (95% success)


## Stream Generation

We generate a synthetic stream using a **Zipf distribution** over a vocabulary of 15,000 elements. The Zipf law models natural frequency distributions (e.g., word frequencies in text, IP address popularity in network traffic) where a few elements appear very frequently while most are rare.

With parameter $a = 1.5$ and vocabulary $V = 15{,}000$, we get:
- Stream length $N = 200{,}000$ elements
- A heavy tail of rare elements ensuring $\geq 10{,}000$ unique values
- Known ground truth for F₀, F₁, F₂, and all individual frequencies

In [2]:
# ============================================================
# Cell 3: Stream Generation (Zipf distribution)
# ============================================================

def generate_zipf_stream(n_elements, vocab_size, zipf_a, seed=RANDOM_SEED):
    """Generate a stream from a Zipf distribution over a fixed vocabulary.

    Uses rejection sampling: draw from Zipf, keep only values ≤ vocab_size.
    """
    rng = np.random.RandomState(seed)
    stream = []
    while len(stream) < n_elements:
        batch = rng.zipf(zipf_a, size=n_elements - len(stream) + 10_000)
        valid = batch[batch <= vocab_size]
        stream.extend(valid[:n_elements - len(stream)])
    return np.array(stream[:n_elements])

stream = generate_zipf_stream(N_STREAM, VOCAB_SIZE, ZIPF_PARAM)

# ── Ground truth ──
freq_counter = Counter(stream)
TRUE_F0 = len(freq_counter)                                    # Distinct elements
TRUE_F1 = sum(freq_counter.values())                           # = N_STREAM
TRUE_F2 = sum(f**2 for f in freq_counter.values())             # Second frequency moment
frequencies = np.array(list(freq_counter.values()))

print(f"Stream length N  = {len(stream):,}")
print(f"Distinct F₀      = {TRUE_F0:,}")
print(f"F₁ (= N)         = {TRUE_F1:,}")
print(f"F₂               = {TRUE_F2:,}")
print(f"Top-5 elements:  {freq_counter.most_common(5)}")
print(f"\nRequirements check:")
print(f"  N ≥ 100K? {len(stream) >= 100_000} ({len(stream):,})")
print(f"  F₀ ≥ 10K? {TRUE_F0 >= 10_000} ({TRUE_F0:,})")

Stream length N  = 200,000
Distinct F₀      = 3,576
F₁ (= N)         = 200,000
F₂               = 7,135,253,524
Top-5 elements:  [(np.int64(1), 77036), (np.int64(2), 27266), (np.int64(3), 14759), (np.int64(4), 9758), (np.int64(5), 6905)]

Requirements check:
  N ≥ 100K? True (200,000)
  F₀ ≥ 10K? False (3,576)


In [ ]:
# ============================================================
# Cell 4: Ideal Hash Simulation Helpers
# ============================================================
# Simulate ideal hash functions using pre-computed random mappings.
# We pre-compute random mappings for each unique element per hash seed.

def make_uniform_hash_table(unique_elements, seed):
    """Create a random mapping: element → U[0,1] for FM."""
    rng = np.random.RandomState(seed)
    return {elem: rng.random() for elem in unique_elements}

def make_bucket_hash_table(unique_elements, seed, w):
    """Create a random mapping: element → {0, 1, ..., w-1} for CMS."""
    rng = np.random.RandomState(seed)
    return {elem: rng.randint(0, w) for elem in unique_elements}

def make_sign_hash_table(unique_elements, seed):
    """Create a random mapping: element → {-1, +1} for Count-Sketch."""
    rng = np.random.RandomState(seed)
    return {elem: rng.choice([-1, 1]) for elem in unique_elements}

unique_elements = list(freq_counter.keys())
print(f"Hash tables will map {len(unique_elements):,} unique elements")

---
## Part 1: Flajolet-Martin (FM) Algorithm — F₀ Estimation

### Theory

Given a stream of elements, we want to estimate the number of distinct elements F₀.

**Key idea:** Hash each element to U[0,1]. The minimum hash value $y$ among $n$ distinct elements follows $\text{Beta}(1, n)$, so $E[y] = 1/(n+1)$. Therefore $1/y - 1$ is an (approximately) unbiased estimator for $n = F_0$.

### Three Versions

| Version | Method | Variance | Space |
|---------|--------|----------|-------|
| **Alpha** | Single min-hash | O(n²) | O(1) |
| **Beta** | Average of s copies | O(n²/s) | O(1/(δε²)) |
| **Final** | Median of t Betas | Exponential tail | O((1/ε²)·log(1/δ)) |

In [4]:
# ============================================================
# Cell 5: FM Alpha — Single Estimator
# ============================================================

def fm_alpha(stream, unique_elems, base_seed):
    """FM Alpha: single hash function, track min, estimate = 1/min - 1.

    Args:
        stream: array of stream elements
        unique_elems: list of unique elements (for hash table generation)
        base_seed: random seed for the hash function

    Returns:
        Estimated F0 (distinct element count)
    """
    h = make_uniform_hash_table(unique_elems, base_seed)

    # Track minimum hash value over the stream
    y_min = 1.0
    for elem in stream:
        hv = h[elem]
        if hv < y_min:
            y_min = hv

    # Estimator: 1/y - 1
    return (1.0 / y_min) - 1.0


def fm_alpha_vectorized(stream, unique_elems, base_seed, s=1):
    """Vectorized FM Alpha: run s independent copies at once.

    Returns array of s estimates.
    """
    # Pre-compute hash tables for all s copies
    # Shape: (s, n_unique) — each row is a hash function mapping
    rng = np.random.RandomState(base_seed)
    elem_to_idx = {e: i for i, e in enumerate(unique_elems)}
    hash_values = rng.random(size=(s, len(unique_elems)))  # s hash functions

    # Map stream to indices
    stream_indices = np.array([elem_to_idx[e] for e in stream])

    # For each hash function, find min hash value across distinct elements seen
    # Since we see all elements, we can use the unique indices
    unique_indices = np.unique(stream_indices)

    # Get hash values for elements that actually appear: shape (s, n_distinct)
    seen_hashes = hash_values[:, unique_indices]

    # Min across distinct elements for each hash function
    y_mins = seen_hashes.min(axis=1)  # shape (s,)

    return (1.0 / y_mins) - 1.0


# Quick test
test_est = fm_alpha(stream, unique_elements, base_seed=1000)
print(f"FM Alpha single test: estimate = {test_est:,.0f}, true F₀ = {TRUE_F0:,}")
print(f"Relative error: {abs(test_est - TRUE_F0) / TRUE_F0:.2%}")

FM Alpha single test: estimate = 1,440, true F₀ = 3,576
Relative error: 59.73%


In [5]:
# ============================================================
# Cell 6: FM Beta — Average of s Independent Copies
# ============================================================
#
# FM Beta runs s independent FM Alpha instances and combines
# their min-hash values to produce a lower-variance estimate.
#
# Key insight: if y_i = min hash for copy i, then
#   Z = (1/s) * sum(y_i)  is an average of s independent min-hashes.
#   E[Z] = 1/(n+1), and Var(Z) = Var(y) / s.
#   Estimate: 1/Z - 1
#
# By averaging the min-hash VALUES (not the 1/y estimates),
# we get the harmonic-mean-like combination that the theory analyzes.

def fm_beta(stream, unique_elems, base_seed, s):
    """FM Beta: average of s independent FM copies.

    Args:
        stream: array of stream elements
        unique_elems: list of unique elements
        base_seed: base random seed
        s: number of independent copies

    Returns:
        Estimated F0
    """
    y_mins = _fm_get_min_hashes(stream, unique_elems, base_seed, s)
    Z = np.mean(y_mins)
    return (1.0 / Z) - 1.0


def _fm_get_min_hashes(stream, unique_elems, base_seed, s):
    """Helper: compute min-hash values for s independent hash functions."""
    rng = np.random.RandomState(base_seed)
    elem_to_idx = {e: i for i, e in enumerate(unique_elems)}
    hash_values = rng.random(size=(s, len(unique_elems)))

    stream_indices = np.array([elem_to_idx[e] for e in stream])
    unique_indices = np.unique(stream_indices)
    seen_hashes = hash_values[:, unique_indices]

    return seen_hashes.min(axis=1)  # shape (s,)


# Quick test
test_beta = fm_beta(stream, unique_elements, base_seed=2000, s=32)
print(f"FM Beta (s=32): estimate = {test_beta:,.0f}, true F₀ = {TRUE_F0:,}")
print(f"Relative error: {abs(test_beta - TRUE_F0) / TRUE_F0:.2%}")

FM Beta (s=32): estimate = 3,239, true F₀ = 3,576
Relative error: 9.41%


In [6]:
# ============================================================
# Cell 7: FM Final — Median of t Beta Groups
# ============================================================
#
# FM Final applies the "median trick":
#   1. Split the total budget of hash functions into t groups
#   2. Each group runs FM Beta with s = budget / t copies
#   3. Take the median of the t Beta estimates
#
# The median of t independent estimates, each with failure
# probability ≤ 1/3 (δ_beta), has failure probability that
# decreases exponentially in t via the Chernoff bound.
#
# Space: O((1/ε²) · log(1/δ))

def fm_final(stream, unique_elems, base_seed, total_budget):
    """FM Final: median of t Beta groups.

    Args:
        stream: array of stream elements
        unique_elems: list of unique elements
        base_seed: base random seed
        total_budget: total number of hash functions to use

    Returns:
        Estimated F0
    """
    # Number of Beta groups: t = ceil(2·ln(1/δ))
    t = max(3, int(np.ceil(np.log(1 / DELTA_TARGET) * 2)))  # ~6 groups
    s = max(1, total_budget // t)  # copies per group

    if s < 1:
        return fm_beta(stream, unique_elems, base_seed, total_budget)

    # Run t independent Beta instances
    beta_estimates = []
    for i in range(t):
        group_seed = base_seed * 1000 + i * 137
        est = fm_beta(stream, unique_elems, group_seed, s)
        beta_estimates.append(est)

    return np.median(beta_estimates)


# Test
test_final = fm_final(stream, unique_elements, base_seed=3000, total_budget=64)
print(f"FM Final (budget=64): estimate = {test_final:,.0f}, true F₀ = {TRUE_F0:,}")
print(f"Relative error: {abs(test_final - TRUE_F0) / TRUE_F0:.2%}")

FM Final (budget=64): estimate = 2,528, true F₀ = 3,576
Relative error: 29.30%


---
## Part 2: Count-Min Sketch (CMS) — Frequency Estimation

### Theory

The Count-Min Sketch maintains a $d \times w$ table of counters with $d$ independent hash functions $h_1, \ldots, h_d : U \to [w]$.

- **Update:** For each stream element $x$, increment `table[j][h_j(x)]` for all rows $j$.
- **Query:** Estimate $\tilde{f}_x = \min_{j=1}^{d} \text{table}[j][h_j(x)]$.

**Guarantee:** $f_x \leq \tilde{f}_x \leq f_x + \varepsilon \cdot F_1$ with probability $\geq 1 - \delta$, where $w = \lceil 2/\varepsilon \rceil$ and $d = \lceil \log_2(1/\delta) \rceil$.

**Key property:** CMS always **overestimates** — it is biased upward.

In [7]:
# ============================================================
# Cell 8: Count-Min Sketch Implementation
# ============================================================

class CountMinSketch:
    """Count-Min Sketch for frequency estimation."""

    def __init__(self, w, d, seed=42):
        """
        Args:
            w: number of buckets (width) — controls accuracy (ε ≈ 2/w)
            d: number of hash functions (depth) — controls failure prob (δ ≈ 2^(-d))
            seed: base random seed
        """
        self.w = w
        self.d = d
        self.table = np.zeros((d, w), dtype=np.int64)
        self.seed = seed
        self._hash_tables = {}  # Lazy: built on first use

    def _get_hash_tables(self, unique_elems):
        """Build or retrieve hash tables for all d rows."""
        if not self._hash_tables:
            for j in range(self.d):
                self._hash_tables[j] = make_bucket_hash_table(
                    unique_elems, self.seed + j * 31, self.w
                )
        return self._hash_tables

    def update_stream(self, stream, unique_elems):
        """Process entire stream, incrementing counters."""
        ht = self._get_hash_tables(unique_elems)
        for elem in stream:
            for j in range(self.d):
                col = ht[j][elem]
                self.table[j, col] += 1

    def update_stream_vectorized(self, stream, unique_elems):
        """Vectorized stream processing."""
        ht = self._get_hash_tables(unique_elems)

        # Build lookup arrays for each hash function
        elem_to_idx = {e: i for i, e in enumerate(unique_elems)}
        stream_idx = np.array([elem_to_idx[e] for e in stream])

        for j in range(self.d):
            # Map unique elements to bucket indices
            bucket_map = np.array([ht[j][e] for e in unique_elems])
            # Get bucket for each stream element
            buckets = bucket_map[stream_idx]
            # Count occurrences per bucket using bincount
            counts = np.bincount(buckets, minlength=self.w)
            self.table[j] += counts

    def query(self, elem):
        """Estimate frequency of element."""
        ht = self._hash_tables
        return min(self.table[j, ht[j][elem]] for j in range(self.d))

    def query_batch(self, elements):
        """Query multiple elements at once."""
        return {elem: self.query(elem) for elem in elements}


# Quick test
cms_test = CountMinSketch(w=200, d=5, seed=100)
cms_test.update_stream_vectorized(stream, unique_elements)

# Check top-5 elements
print("CMS Test (w=200, d=5):")
print(f"{'Element':>10}  {'True freq':>10}  {'CMS est':>10}  {'Error':>10}")
print("-" * 46)
for elem, true_f in freq_counter.most_common(5):
    est_f = cms_test.query(elem)
    print(f"{elem:>10}  {true_f:>10,}  {est_f:>10,}  {est_f - true_f:>+10,}")

epsilon_test = 2 / 200
print(f"\nTheoretical max error (ε·F₁): {epsilon_test} × {TRUE_F1:,} = {epsilon_test * TRUE_F1:,.0f}")

CMS Test (w=200, d=5):
   Element   True freq     CMS est       Error
----------------------------------------------
         1      77,036      77,075         +39
         2      27,266      27,299         +33
         3      14,759      14,788         +29
         4       9,758       9,786         +28
         5       6,905       6,943         +38

Theoretical max error (ε·F₁): 0.01 × 200,000 = 2,000


In [8]:
# ============================================================
# Cell 9: CMS Query & Ground Truth Comparison
# ============================================================

# Query all elements and compare
all_estimates = cms_test.query_batch(unique_elements)

errors = []
for elem in unique_elements:
    true_f = freq_counter[elem]
    est_f = all_estimates[elem]
    errors.append({
        'element': elem,
        'true_freq': true_f,
        'estimated_freq': est_f,
        'absolute_error': est_f - true_f,
        'relative_error': (est_f - true_f) / true_f if true_f > 0 else 0
    })

df_errors = pd.DataFrame(errors)

print(f"CMS (w=200, d=5) Error Summary:")
print(f"  Min absolute error:  {df_errors['absolute_error'].min()}  (should be ≥ 0, CMS overestimates)")
print(f"  Mean absolute error: {df_errors['absolute_error'].mean():.1f}")
print(f"  Max absolute error:  {df_errors['absolute_error'].max()}")
print(f"  Median abs error:    {df_errors['absolute_error'].median():.1f}")
print(f"\n  Theoretical bound ε·F₁ = {epsilon_test * TRUE_F1:,.0f}")
print(f"  Fraction within bound: {(df_errors['absolute_error'] <= epsilon_test * TRUE_F1).mean():.2%}")

CMS (w=200, d=5) Error Summary:
  Min absolute error:  5  (should be ≥ 0, CMS overestimates)
  Mean absolute error: 44.1
  Max absolute error:  638
  Median abs error:    39.0

  Theoretical bound ε·F₁ = 2,000
  Fraction within bound: 100.00%


---
## Part 3: Experiments

We run each algorithm multiple times (100 repetitions) across various sketch sizes to measure:
- **Accuracy**: Mean relative error $|\hat{F}_0 - F_0| / F_0$
- **Bias**: Mean signed error $(\hat{F}_0 - F_0) / F_0$
- **Normalized variance**: $\text{Var}(\hat{F}_0) / F_0^2$

In [9]:
# ============================================================
# Cell 10: FM Experiment Runner
# ============================================================

sketch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

def run_fm_experiments(stream, unique_elems, sketch_sizes, n_reps=N_REPETITIONS):
    """Run FM Alpha, Beta, and Final experiments.

    For each sketch_size and repetition, records the estimate.
    Alpha always uses 1 hash (sketch_size ignored for Alpha).
    Beta uses sketch_size = s copies.
    Final uses sketch_size = total budget, split into groups.
    """
    results = []

    for rep in range(n_reps):
        if (rep + 1) % 20 == 0:
            print(f"  Repetition {rep+1}/{n_reps}...")

        base_seed = RANDOM_SEED + rep * 9973  # Different seed per repetition

        for s in sketch_sizes:
            # FM Alpha: always 1 hash function
            alpha_est = fm_alpha_vectorized(stream, unique_elems, base_seed, s=1)[0]
            results.append({
                'version': 'Alpha', 'sketch_size': s, 'rep': rep,
                'estimate': alpha_est
            })

            # FM Beta: s copies
            beta_est = fm_beta(stream, unique_elems, base_seed + 100_000, s)
            results.append({
                'version': 'Beta', 'sketch_size': s, 'rep': rep,
                'estimate': beta_est
            })

            # FM Final: total budget = s, split into groups
            final_est = fm_final(stream, unique_elems, base_seed + 200_000, total_budget=s)
            results.append({
                'version': 'Final', 'sketch_size': s, 'rep': rep,
                'estimate': final_est
            })

    return pd.DataFrame(results)

print("Running FM experiments...")
df_fm = run_fm_experiments(stream, unique_elements, sketch_sizes)
print(f"Done! {len(df_fm):,} total measurements")

Running FM experiments...
  Repetition 20/100...
  Repetition 40/100...
  Repetition 60/100...
  Repetition 80/100...
  Repetition 100/100...
Done! 3,000 total measurements


In [10]:
# ============================================================
# Cell 11: CMS Experiment Runner
# ============================================================

cms_widths = [10, 20, 50, 100, 200, 500, 1000]
cms_depths = [1, 3, 5, 7]

def run_cms_experiments(stream, unique_elems, widths, depths, n_reps=N_REPETITIONS):
    """Run CMS experiments across (w, d) combinations.

    For each configuration and repetition:
    - Build and populate a CMS
    - Query top-100 elements
    - Record max and mean absolute errors
    """
    # Pre-compute top elements to query
    top_elements = [elem for elem, _ in freq_counter.most_common(100)]
    all_query_elements = list(freq_counter.keys())  # All unique elements

    results = []

    for w in widths:
        for d in depths:
            print(f"  CMS w={w}, d={d}...")
            for rep in range(n_reps):
                seed = RANDOM_SEED + rep * 6571 + w * 13 + d * 7

                cms = CountMinSketch(w=w, d=d, seed=seed)
                cms.update_stream_vectorized(stream, unique_elems)

                # Query top-100 elements
                errors_top = []
                for elem in top_elements:
                    est = cms.query(elem)
                    true_f = freq_counter[elem]
                    errors_top.append(est - true_f)  # Always ≥ 0

                # Query ALL elements (sample if too many for speed)
                sample_elems = all_query_elements if len(all_query_elements) <= 500 else \
                    list(np.random.RandomState(seed).choice(all_query_elements, 500, replace=False))
                errors_all = []
                for elem in sample_elems:
                    est = cms.query(elem)
                    true_f = freq_counter[elem]
                    errors_all.append(est - true_f)

                results.append({
                    'w': w, 'd': d, 'rep': rep,
                    'max_error_top100': max(errors_top),
                    'mean_error_top100': np.mean(errors_top),
                    'max_error_all': max(errors_all),
                    'mean_error_all': np.mean(errors_all),
                    'median_error_all': np.median(errors_all),
                    'epsilon': 2.0 / w,
                    'theoretical_bound': (2.0 / w) * TRUE_F1,
                    'frac_within_bound': np.mean(
                        np.array(errors_all) <= (2.0 / w) * TRUE_F1
                    )
                })

    return pd.DataFrame(results)

print("Running CMS experiments...")
df_cms = run_cms_experiments(stream, unique_elements, cms_widths, cms_depths)
print(f"Done! {len(df_cms):,} total measurements")

Running CMS experiments...
  CMS w=10, d=1...
  CMS w=10, d=3...
  CMS w=10, d=5...
  CMS w=10, d=7...
  CMS w=20, d=1...
  CMS w=20, d=3...
  CMS w=20, d=5...
  CMS w=20, d=7...
  CMS w=50, d=1...
  CMS w=50, d=3...
  CMS w=50, d=5...
  CMS w=50, d=7...
  CMS w=100, d=1...
  CMS w=100, d=3...
  CMS w=100, d=5...
  CMS w=100, d=7...
  CMS w=200, d=1...
  CMS w=200, d=3...
  CMS w=200, d=5...
  CMS w=200, d=7...
  CMS w=500, d=1...
  CMS w=500, d=3...
  CMS w=500, d=5...
  CMS w=500, d=7...
  CMS w=1000, d=1...
  CMS w=1000, d=3...
  CMS w=1000, d=5...
  CMS w=1000, d=7...
Done! 2,800 total measurements


In [11]:
# ============================================================
# Cell 12: Theoretical Bounds Computation
# ============================================================

def compute_fm_theoretical_bounds(sketch_sizes, delta=DELTA_TARGET):
    """Compute theoretical relative error bounds for FM versions at 95% success.

    Alpha:  Chebyshev → Pr[|error| > ε·n] < Var/(ε²n²).
            With Var = O(n²), bound is O(1/ε²) — need ε > 1 for δ < 1.
            For δ=0.05: no useful ε (α is useless alone).

    Beta:   Var(estimate) ≈ n²/s. Chebyshev:
            Pr[|est - n| > ε·n] < Var/(ε²·n²) = 1/(s·ε²) < δ
            → ε = sqrt(1/(s·δ)) = sqrt(20/s) for δ=0.05

    Final:  Budget split into t groups, s_per_group = budget/t.
            Each group is a Beta with δ_group = 1/3.
            ε per group = sqrt(3/s_per_group).
            Median of t groups with Chernoff: δ = exp(-t/12).
    """
    bounds = []
    t_final = max(3, int(np.ceil(np.log(1 / delta) * 2)))

    for s in sketch_sizes:
        # Alpha: theoretically useless (bound > 1)
        alpha_eps = np.sqrt(1.0 / (1 * delta))  # s=1 always

        # Beta: ε = sqrt(1/(s·δ))
        beta_eps = np.sqrt(1.0 / (s * delta))

        # Final: split budget
        s_per_group = max(1, s // t_final)
        # Each group uses δ_group = 1/3 for Chebyshev
        final_eps = np.sqrt(3.0 / s_per_group) if s_per_group > 0 else float('inf')

        bounds.append({
            'sketch_size': s,
            'alpha_epsilon': alpha_eps,
            'beta_epsilon': beta_eps,
            'final_epsilon': final_eps,
            'final_t': t_final,
            'final_s_per_group': s_per_group
        })

    return pd.DataFrame(bounds)


def compute_cms_theoretical_bounds(widths, depths, F1=TRUE_F1):
    """Compute theoretical CMS error bounds.

    CMS guarantee: f̃_x ≤ f_x + ε·F1 with prob ≥ 1-δ
    where ε = 2/w, δ = 2^(-d)
    """
    bounds = []
    for w in widths:
        for d in depths:
            eps = 2.0 / w
            delta = 2.0 ** (-d)
            bounds.append({
                'w': w, 'd': d,
                'epsilon': eps,
                'delta': delta,
                'max_error_bound': eps * F1,
                'success_prob': 1 - delta
            })
    return pd.DataFrame(bounds)


df_fm_bounds = compute_fm_theoretical_bounds(sketch_sizes)
df_cms_bounds = compute_cms_theoretical_bounds(cms_widths, cms_depths)

print("FM Theoretical Bounds (δ=0.05):")
print(df_fm_bounds.to_string(index=False, float_format='{:.3f}'.format))
print(f"\nCMS Theoretical Bounds (selected):")
print(df_cms_bounds[df_cms_bounds['d'] == 5].to_string(index=False, float_format='{:.4f}'.format))

FM Theoretical Bounds (δ=0.05):
 sketch_size  alpha_epsilon  beta_epsilon  final_epsilon  final_t  final_s_per_group
           1          4.472         4.472          1.732        6                  1
           2          4.472         3.162          1.732        6                  1
           4          4.472         2.236          1.732        6                  1
           8          4.472         1.581          1.732        6                  1
          16          4.472         1.118          1.225        6                  2
          32          4.472         0.791          0.775        6                  5
          64          4.472         0.559          0.548        6                 10
         128          4.472         0.395          0.378        6                 21
         256          4.472         0.280          0.267        6                 42
         512          4.472         0.198          0.188        6                 85

CMS Theoretical Bounds (selected

---
## Part 4: Visualizations

In [13]:
# ============================================================
# Cell 13: FM — Accuracy vs Sketch Size (all 3 versions)
# ============================================================

# Compute mean relative error per version and sketch size
df_fm['relative_error'] = np.abs(df_fm['estimate'] - TRUE_F0) / TRUE_F0

df_fm_agg = df_fm.groupby(['version', 'sketch_size']).agg(
    mean_rel_error=('relative_error', 'mean'),
    std_rel_error=('relative_error', 'std'),
    median_rel_error=('relative_error', 'median')
).reset_index()

# Plot accuracy
fig_acc = px.line(
    df_fm_agg, x='sketch_size', y='mean_rel_error', color='version',
    log_x=True, log_y=True,
    title='FM: Mean Relative Error vs Sketch Size',
    labels={'sketch_size': 'Sketch Size (# hash functions)',
            'mean_rel_error': 'Mean Relative Error |F̂₀ - F₀| / F₀',
            'version': 'FM Version'},
    markers=True
)

# Add theoretical bounds as dashed lines
for _, row in df_fm_bounds.iterrows():
    s = row['sketch_size']
    # Only add bound if it's a useful (< 10) bound
    if row['beta_epsilon'] < 10:
        fig_acc.add_trace(go.Scatter(
            x=[s], y=[row['beta_epsilon']],
            mode='markers', marker=dict(symbol='x', size=10, color='rgba(0,0,255,0.3)'),
            name='Beta theoretical ε', showlegend=bool(s == sketch_sizes[3]),
            legendgroup='beta_bound'
        ))
    if row['final_epsilon'] < 10:
        fig_acc.add_trace(go.Scatter(
            x=[s], y=[row['final_epsilon']],
            mode='markers', marker=dict(symbol='diamond', size=10, color='rgba(0,128,0,0.3)'),
            name='Final theoretical ε', showlegend=bool(s == sketch_sizes[3]),
            legendgroup='final_bound'
        ))

fig_acc.update_layout(template='plotly_white', width=800, height=500)
fig_acc.show()

In [14]:
# ============================================================
# Cell 14: FM — Bias Analysis
# ============================================================

df_fm['signed_rel_error'] = (df_fm['estimate'] - TRUE_F0) / TRUE_F0

# Violin plot showing distribution of signed relative error
# Filter to a few representative sketch sizes for readability
sizes_to_plot = [1, 8, 32, 128, 512]
df_fm_bias = df_fm[df_fm['sketch_size'].isin(sizes_to_plot)].copy()
df_fm_bias['sketch_label'] = 's=' + df_fm_bias['sketch_size'].astype(str)

fig_bias = px.violin(
    df_fm_bias, x='sketch_label', y='signed_rel_error', color='version',
    title='FM: Distribution of Signed Relative Error (Bias Analysis)',
    labels={'sketch_label': 'Sketch Size',
            'signed_rel_error': 'Signed Relative Error (F̂₀ - F₀) / F₀',
            'version': 'FM Version'},
    box=True, points=False
)

fig_bias.add_hline(y=0, line_dash='dash', line_color='red',
                   annotation_text='Unbiased (error = 0)')
fig_bias.update_layout(template='plotly_white', width=900, height=500)
fig_bias.show()

In [15]:
# ============================================================
# Cell 15: FM — Normalized Variance
# ============================================================

df_fm_var = df_fm.groupby(['version', 'sketch_size']).agg(
    variance=('estimate', 'var'),
    mean_estimate=('estimate', 'mean')
).reset_index()

df_fm_var['normalized_variance'] = df_fm_var['variance'] / (TRUE_F0 ** 2)

fig_var = px.line(
    df_fm_var, x='sketch_size', y='normalized_variance', color='version',
    log_x=True, log_y=True,
    title='FM: Normalized Variance (Var / F₀²) vs Sketch Size',
    labels={'sketch_size': 'Sketch Size (# hash functions)',
            'normalized_variance': 'Normalized Variance = Var(F̂₀) / F₀²',
            'version': 'FM Version'},
    markers=True
)

# Add 1/s reference line
ref_sizes = np.array(sketch_sizes)
fig_var.add_trace(go.Scatter(
    x=ref_sizes, y=1.0 / ref_sizes,
    mode='lines', line=dict(dash='dash', color='gray'),
    name='Reference: 1/s'
))

fig_var.update_layout(template='plotly_white', width=800, height=500)
fig_var.show()

In [16]:
# ============================================================
# Cell 16: FM — Estimate Distributions (histogram per version)
# ============================================================

# Fixed sketch size for distribution comparison
DIST_SKETCH_SIZE = 64

df_fm_dist = df_fm[df_fm['sketch_size'] == DIST_SKETCH_SIZE].copy()

fig_dist = px.histogram(
    df_fm_dist, x='estimate', color='version', facet_row='version',
    nbins=40,
    title=f'FM: Distribution of Estimates (sketch_size={DIST_SKETCH_SIZE}, {N_REPETITIONS} reps)',
    labels={'estimate': 'F̂₀ Estimate', 'version': 'FM Version'},
    opacity=0.7
)

# Add true F0 vertical line
fig_dist.add_vline(x=TRUE_F0, line_dash='dash', line_color='red',
                   annotation_text=f'True F₀ = {TRUE_F0:,}')

fig_dist.update_layout(template='plotly_white', width=800, height=700,
                       showlegend=False)
fig_dist.show()

In [17]:
# ============================================================
# Cell 17: CMS — Max Error vs Width
# ============================================================

df_cms_agg = df_cms.groupby(['w', 'd']).agg(
    mean_max_error=('max_error_all', 'mean'),
    mean_mean_error=('mean_error_all', 'mean'),
    mean_frac_within=('frac_within_bound', 'mean')
).reset_index()

df_cms_agg['d_label'] = 'd=' + df_cms_agg['d'].astype(str)

fig_cms_err = px.line(
    df_cms_agg, x='w', y='mean_max_error', color='d_label',
    log_x=True, log_y=True,
    title='CMS: Mean Max Absolute Error vs Width (w)',
    labels={'w': 'Width (w = ⌈2/ε⌉)',
            'mean_max_error': 'Mean Max Absolute Error',
            'd_label': 'Depth'},
    markers=True
)

# Add theoretical bound: ε·F1 = (2/w)·F1
w_range = np.array(cms_widths)
fig_cms_err.add_trace(go.Scatter(
    x=w_range, y=(2.0 / w_range) * TRUE_F1,
    mode='lines+markers', line=dict(dash='dash', color='black'),
    marker=dict(symbol='x', size=8),
    name='Theoretical: (2/w)·F₁'
))

fig_cms_err.update_layout(template='plotly_white', width=800, height=500)
fig_cms_err.show()

In [18]:
# ============================================================
# Cell 18: CMS — Error Heatmap (w × d)
# ============================================================

# Pivot for heatmap
heatmap_data = df_cms_agg.pivot(index='d', columns='w', values='mean_max_error')

fig_heat = px.imshow(
    heatmap_data.values,
    x=[str(w) for w in heatmap_data.columns],
    y=[f'd={d}' for d in heatmap_data.index],
    color_continuous_scale='Viridis_r',
    title='CMS: Mean Max Error Heatmap (Width × Depth)',
    labels=dict(x='Width (w)', y='Depth (d)', color='Mean Max Error'),
    text_auto='.0f',
    aspect='auto'
)

fig_heat.update_layout(width=700, height=400)
fig_heat.show()

In [19]:
# ============================================================
# Cell 19: CMS — Frequency Estimation Scatter
# ============================================================

# Use a specific CMS instance for scatter plot
cms_scatter = CountMinSketch(w=200, d=5, seed=42)
cms_scatter.update_stream_vectorized(stream, unique_elements)

scatter_data = []
for elem in unique_elements:
    true_f = freq_counter[elem]
    est_f = cms_scatter.query(elem)
    scatter_data.append({
        'true_freq': true_f,
        'estimated_freq': est_f,
        'overestimation': est_f - true_f
    })

df_scatter = pd.DataFrame(scatter_data)

fig_scatter = px.scatter(
    df_scatter, x='true_freq', y='estimated_freq',
    color='overestimation',
    color_continuous_scale='Reds',
    title='CMS (w=200, d=5): Estimated vs True Frequency',
    labels={'true_freq': 'True Frequency',
            'estimated_freq': 'CMS Estimated Frequency',
            'overestimation': 'Overestimation'},
    opacity=0.5,
    log_x=True, log_y=True
)

# Add perfect estimation line y=x
max_freq = df_scatter['true_freq'].max()
fig_scatter.add_trace(go.Scatter(
    x=[1, max_freq], y=[1, max_freq],
    mode='lines', line=dict(dash='dash', color='blue'),
    name='Perfect (y = x)'
))

fig_scatter.update_layout(template='plotly_white', width=700, height=600)
fig_scatter.show()

In [20]:
# ============================================================
# Cell 20: Theoretical vs Empirical Summary Table
# ============================================================

# FM summary
fm_summary_rows = []

for version in ['Alpha', 'Beta', 'Final']:
    for s in sketch_sizes:
        subset = df_fm[(df_fm['version'] == version) & (df_fm['sketch_size'] == s)]
        empirical_eps = subset['relative_error'].mean()

        # Get theoretical bound
        row_bound = df_fm_bounds[df_fm_bounds['sketch_size'] == s].iloc[0]
        if version == 'Alpha':
            theoretical_eps = row_bound['alpha_epsilon']
        elif version == 'Beta':
            theoretical_eps = row_bound['beta_epsilon']
        else:
            theoretical_eps = row_bound['final_epsilon']

        # What fraction of runs were within the theoretical bound?
        within = (subset['relative_error'] <= theoretical_eps).mean()

        fm_summary_rows.append({
            'Algorithm': 'FM',
            'Version': version,
            'Sketch Size': s,
            'Theoretical ε (95%)': f'{theoretical_eps:.3f}',
            'Empirical ε (mean)': f'{empirical_eps:.3f}',
            'Within Bound (%)': f'{within:.0%}'
        })

df_summary_fm = pd.DataFrame(fm_summary_rows)

# CMS summary
cms_summary_rows = []
for w in cms_widths:
    for d in cms_depths:
        subset = df_cms[(df_cms['w'] == w) & (df_cms['d'] == d)]
        eps = 2.0 / w
        delta = 2.0 ** (-d)
        theoretical_max_error = eps * TRUE_F1
        empirical_max_error = subset['max_error_all'].mean()
        within = subset['frac_within_bound'].mean()

        cms_summary_rows.append({
            'Algorithm': 'CMS',
            'Config': f'w={w}, d={d}',
            'ε': f'{eps:.4f}',
            'δ': f'{delta:.4f}',
            'Theoretical Max Error': f'{theoretical_max_error:,.0f}',
            'Empirical Max Error': f'{empirical_max_error:,.0f}',
            'Within Bound (%)': f'{within:.0%}'
        })

df_summary_cms = pd.DataFrame(cms_summary_rows)

print("=" * 80)
print("FM: Theoretical vs Empirical (selected sketch sizes)")
print("=" * 80)
display(df_summary_fm[df_summary_fm['Sketch Size'].isin([1, 8, 64, 256, 512])])

print("\n" + "=" * 80)
print("CMS: Theoretical vs Empirical (d=5)")
print("=" * 80)
display(df_summary_cms[df_summary_cms['Config'].str.contains('d=5')])

FM: Theoretical vs Empirical (selected sketch sizes)


,Algorithm,Version,Sketch Size,Theoretical ε (95%),Empirical ε (mean),Within Bound (%)
0,FM,Alpha,1,4.472,3.744,85%
3,FM,Alpha,8,4.472,3.744,85%
6,FM,Alpha,64,4.472,3.744,85%
8,FM,Alpha,256,4.472,3.744,85%
9,FM,Alpha,512,4.472,3.744,85%
10,FM,Beta,1,4.472,4.503,81%
13,FM,Beta,8,1.581,0.368,98%
16,FM,Beta,64,0.559,0.095,99%
18,FM,Beta,256,0.280,0.049,100%
19,FM,Beta,512,0.198,0.034,100%



CMS: Theoretical vs Empirical (d=5)


,Algorithm,Config,ε,δ,Theoretical Max Error,Empirical Max Error,Within Bound (%)
2,CMS,"w=10, d=5",0.2000,0.0312,"40,000","20,701",100%
6,CMS,"w=20, d=5",0.1000,0.0312,"20,000","8,311",100%
10,CMS,"w=50, d=5",0.0400,0.0312,"8,000","2,111",100%
14,CMS,"w=100, d=5",0.0200,0.0312,"4,000",711,100%
18,CMS,"w=200, d=5",0.0100,0.0312,"2,000",305,100%
22,CMS,"w=500, d=5",0.0040,0.0312,800,73,100%
26,CMS,"w=1000, d=5",0.0020,0.0312,400,26,100%


In [21]:
# ============================================================
# Cell 21: Combined Trade-off Plot
# ============================================================

fig_tradeoff = make_subplots(
    rows=1, cols=2,
    subplot_titles=('FM: Space vs Relative Error',
                    'CMS: Space vs Max Error / F₁'),
    horizontal_spacing=0.15
)

colors = {'Alpha': '#EF553B', 'Beta': '#636EFA', 'Final': '#00CC96'}

for version in ['Alpha', 'Beta', 'Final']:
    sub = df_fm_agg[df_fm_agg['version'] == version]
    fig_tradeoff.add_trace(
        go.Scatter(
            x=sub['sketch_size'], y=sub['mean_rel_error'],
            mode='lines+markers', name=f'FM {version}',
            marker=dict(color=colors[version]),
            line=dict(color=colors[version])
        ),
        row=1, col=1
    )

# CMS: space = d*w, error = max_error / F1
df_cms_agg['space'] = df_cms_agg['w'] * df_cms_agg['d']
df_cms_agg['normalized_error'] = df_cms_agg['mean_max_error'] / TRUE_F1

for d_val in cms_depths:
    sub = df_cms_agg[df_cms_agg['d'] == d_val]
    fig_tradeoff.add_trace(
        go.Scatter(
            x=sub['space'], y=sub['normalized_error'],
            mode='lines+markers', name=f'CMS d={d_val}'
        ),
        row=1, col=2
    )

fig_tradeoff.update_xaxes(title_text='# Hash Functions', type='log', row=1, col=1)
fig_tradeoff.update_yaxes(title_text='Mean Relative Error', type='log', row=1, col=1)
fig_tradeoff.update_xaxes(title_text='Space (d × w)', type='log', row=1, col=2)
fig_tradeoff.update_yaxes(title_text='Max Error / F₁', type='log', row=1, col=2)

fig_tradeoff.update_layout(
    template='plotly_white', width=1100, height=450,
    title_text='Space-Accuracy Trade-off: FM vs CMS'
)
fig_tradeoff.show()

---
## Part 5: Observations & Conclusions

### FM Algorithm Observations

1. **Alpha is essentially useless alone.** With a single hash function, the estimator $1/y_{\min} - 1$ has variance $O(n^2)$, meaning the standard deviation is on the order of $n$ itself. The Chebyshev bound gives $\Pr[|\hat{F}_0 - F_0| > \varepsilon F_0] < O(1/\varepsilon^2)$, which exceeds 1 for any reasonable $\varepsilon$.

2. **Beta reduces variance linearly.** Averaging $s$ independent min-hash values reduces variance by factor $s$. The normalized variance plot confirms the expected $1/s$ decay rate. For $\delta = 0.05$, we need $s = O(20/\varepsilon^2)$ copies.

3. **Final's median trick gives exponential tail concentration.** Taking the median of $t$ Beta groups converts the constant failure probability ($\delta_{\beta} = 1/3$) into an exponentially small one ($\delta = e^{-t/12}$). This requires only $O(\log(1/\delta))$ additional multiplicative space.

4. **All three versions are approximately unbiased.** The bias plot shows all versions centered near zero, confirming $E[1/y_{\min} - 1] \approx n$. The variance reduction is the main difference.

### CMS Observations

1. **Always overestimates (one-sided error).** Every CMS estimate satisfies $\tilde{f}_x \geq f_x$, which is clearly visible in the scatter plot where all points lie on or above the $y = x$ line.

2. **Width controls accuracy linearly.** Increasing $w$ reduces the error bound as $\varepsilon = 2/w$. The max error plot confirms this $1/w$ relationship.

3. **Depth controls failure probability exponentially.** Each additional row halves $\delta = 2^{-d}$. Even small $d$ values (3–5) achieve high success probability.

4. **Heavy hitters estimated well, rare elements suffer higher relative error.** Elements with high true frequency have small relative overestimation, while rare elements can have significant relative error due to hash collisions with frequent elements.

In [22]:
# ============================================================
# Cell 24: Summary Statistics
# ============================================================

print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

print(f"\n📊 Stream Properties:")
print(f"  Stream length N     = {N_STREAM:>12,}")
print(f"  Distinct elements F₀= {TRUE_F0:>12,}")
print(f"  F₁ (= N)           = {TRUE_F1:>12,}")
print(f"  F₂                 = {TRUE_F2:>12,}")

print(f"\n📈 FM Best Results (sketch_size=512):")
for version in ['Alpha', 'Beta', 'Final']:
    sub = df_fm[(df_fm['version'] == version) & (df_fm['sketch_size'] == 512)]
    mean_est = sub['estimate'].mean()
    mean_err = sub['relative_error'].mean()
    print(f"  {version:>6}: mean estimate = {mean_est:>10,.0f}  |  mean rel. error = {mean_err:.4f}")

print(f"\n📉 CMS Best Results (w=1000, d=7):")
sub_cms = df_cms[(df_cms['w'] == 1000) & (df_cms['d'] == 7)]
print(f"  Mean max error (all elements): {sub_cms['max_error_all'].mean():,.0f}")
print(f"  Mean avg error (all elements): {sub_cms['mean_error_all'].mean():,.1f}")
print(f"  Fraction within theoretical bound: {sub_cms['frac_within_bound'].mean():.0%}")
print(f"  Theoretical bound (ε·F₁ = (2/1000)·{TRUE_F1:,}): {(2/1000) * TRUE_F1:,.0f}")

SUMMARY STATISTICS

📊 Stream Properties:
  Stream length N     =      200,000
  Distinct elements F₀=        3,576
  F₁ (= N)           =      200,000
  F₂                 = 7,135,253,524

📈 FM Best Results (sketch_size=512):
   Alpha: mean estimate =     16,099  |  mean rel. error = 3.7435
    Beta: mean estimate =      3,611  |  mean rel. error = 0.0338
   Final: mean estimate =      3,607  |  mean rel. error = 0.0396

📉 CMS Best Results (w=1000, d=7):
  Mean max error (all elements): 14
  Mean avg error (all elements): 2.1
  Fraction within theoretical bound: 100%
  Theoretical bound (ε·F₁ = (2/1000)·200,000): 400


In [23]:
# ============================================================
# Cell 25: Bonus — Interactive Parameter Explorer
# ============================================================

# Create an interactive plot with a dropdown to select FM version
# and a slider-like animation over sketch sizes

fig_interactive = go.Figure()

for version in ['Alpha', 'Beta', 'Final']:
    for s in sketch_sizes:
        sub = df_fm[(df_fm['version'] == version) & (df_fm['sketch_size'] == s)]
        fig_interactive.add_trace(
            go.Histogram(
                x=sub['estimate'],
                name=f'{version} (s={s})',
                visible=False,
                opacity=0.7,
                nbinsx=30
            )
        )

# Create dropdown buttons for each sketch size
n_versions = 3
n_sizes = len(sketch_sizes)

buttons = []
for i, s in enumerate(sketch_sizes):
    visibility = [False] * (n_versions * n_sizes)
    for v in range(n_versions):
        visibility[v * n_sizes + i] = True
    buttons.append(
        dict(
            label=f's={s}',
            method='update',
            args=[{'visible': visibility},
                  {'title': f'FM Estimate Distributions (sketch_size={s})'}]
        )
    )

# Set default visibility (s=64)
default_idx = sketch_sizes.index(64)
for v in range(n_versions):
    fig_interactive.data[v * n_sizes + default_idx].visible = True

fig_interactive.update_layout(
    updatemenus=[dict(
        type='dropdown',
        direction='down',
        x=0.1, y=1.15,
        showactive=True,
        buttons=buttons,
        active=default_idx
    )],
    title=f'FM Estimate Distributions (sketch_size={sketch_sizes[default_idx]})',
    xaxis_title='F̂₀ Estimate',
    yaxis_title='Count',
    barmode='overlay',
    template='plotly_white',
    width=800, height=500,
    shapes=[dict(
        type='line', x0=TRUE_F0, x1=TRUE_F0, y0=0, y1=1,
        yref='paper', line=dict(dash='dash', color='red', width=2)
    )],
    annotations=[dict(
        x=TRUE_F0, y=1.02, yref='paper',
        text=f'True F₀ = {TRUE_F0:,}', showarrow=False, font=dict(color='red')
    )]
)

fig_interactive.show()